# 📝 하이브리드 검색 과제 LV1(기초)

교안 01의 하이브리드 검색을 한 단계씩 연습합니다. 한국어 백과 문서 5개(`lv1_docs.json`)로 BM25와 Dense 검색기를 만들고, `EnsembleRetriever`로 합친 뒤 같은 후보에서 가중치를 비교합니다.

- 1~3번: 한국어 토큰화, BM25 검색기, BM25 점수 읽기
- 4~5번: Dense 검색기, `EnsembleRetriever` 하이브리드 검색기
- 6번: 같은 후보에서 가중치만 변경(결과 조립·Recall@2 계산·출력 제공)
- 7번: RRF가 원래 점수 대신 순위를 쓰는 이유 설명

문서가 5개이므로 BM25와 Dense는 각각 최대 2개를 가져옵니다. `.env`의 OpenAI 키가 필요합니다. 임베딩만 요청하고(문서 5개와 검색 질문) GPT는 호출하지 않습니다.

**풀이 방법**: 준비 셀부터 순서대로 실행하세요. 구분선 안의 `[작성]` 부분에서 검색기 생성과 핵심 처리 코드를 작성합니다. `...`는 호출식 전체나 여러 줄의 코드로 바꿀 수 있습니다. 입력 준비·결과 조립·비교·출력 코드는 완성되어 있습니다. 문항별 핵심 동작만 자가채점하며, 검색 순위·Recall 수치·모델의 문장 표현은 고정하지 않습니다. 앞 문항의 검색기와 결과를 다음 문항에서 이어 씁니다.


## 준비와 데이터 살펴보기

노트북이 있는 폴더에서 새 커널로 시작하고 준비 셀을 위에서부터 실행하세요. 경로(`material_dir`·`data_dir`·`output_dir`)와 `read_json`·`save_json`은 앞 단원과 같습니다. 첫 셀에서 `langchain-community` 유지보수 종료를 알리는 경고가 한 번 보일 수 있습니다. `BM25Retriever`를 이 패키지에서 가져오기 때문이며 실행에는 문제가 없습니다.


In [ ]:
# 문서·검색기·모델에 필요한 라이브러리를 가져옵니다.
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from kiwipiepy import Kiwi
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings


In [ ]:
# 실행할 노트북 폴더를 기준으로 입력 data와 생성 결과 output을 구분합니다.
material_dir = Path(".")
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)


def read_json(name):
    """data 폴더의 JSON 파일을 목록 또는 딕셔너리로 읽습니다."""
    return json.loads((data_dir / name).read_text(encoding="utf-8"))


def save_json(name, value):
    """처리 결과를 output 폴더에 한글을 유지해 저장합니다."""
    (output_dir / name).write_text(
        json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8"
    )


임베딩은 `text-embedding-3-large`의 768차원입니다. `check_embedding_ctx_length=False`는 자동 길이 검사·분할을 끕니다. API 키는 `.env`에서 읽습니다.


In [ ]:
# 현재 작업 폴더부터 상위로 .env를 찾아 키를 읽습니다.
load_dotenv(find_dotenv(usecwd=True))

# 문서와 질문을 같은 모델·차원으로 바꿔야 같은 인덱스에서 거리를 비교할 수 있습니다.
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large",
    dimensions=768,
    check_embedding_ctx_length=False,  # 자동 길이 검사·분할을 끄고 준비한 짧은 문서를 보냅니다.
)
print("모델 연결 설정 완료. 임베딩은 적재·검색 셀에서 요청합니다.")


절 하나가 이미 짧은 발췌라서 다시 나누지 않고 레코드 하나를 검색 단위 하나로 씁니다. 이 단위를 고정한 채 검색 방식과 질문을 바꿔 결과를 비교합니다. `make_documents`는 원문 ID·제목·출처에 필터용 메타데이터를 더해 `Document`를 만듭니다.


In [ ]:
def make_documents(records):
    """원문 기록의 본문·출처와 검색 조건을 LangChain Document로 바꿉니다."""
    # id는 저장소가, metadata의 source_id는 RRF가 원문을 구별할 때 읽습니다.
    # 두 위치에 같은 원문 ID를 사용하고 출처·필터 메타데이터를 담습니다.
    return [Document(
        id=record["doc_id"],
        page_content=record["text"],
        metadata={"source_id": record["doc_id"], "title": record["title"],
                  "url": record["url"], "source": record["source"], **record["metadata"]},
    ) for record in records]


In [ ]:
# 전체 목록을 먼저 보고 질문에 필요한 필터 필드를 확인합니다.
records = read_json("lv1_docs.json")
display(pd.DataFrame(records)[["doc_id", "title", "metadata"]])
print(records[0]["text"])
documents = make_documents(records)


In [ ]:
def show_results(documents):
    """앞 5개 결과의 원문 ID·메타데이터·본문을 모든 열과 함께 보여 줍니다."""
    # 빈 결과는 필터를 바꾸지 않고 그대로 알립니다.
    if not documents:
        print("조건에 맞는 검색 결과가 없습니다.")
        return
    # 합집합·조건 목록에서도 쓰므로 번호를 관련성 순위라고 부르지 않습니다.
    # 열 순서를 고정해야 표를 나란히 비교할 수 있습니다. url은 길어서 뺍니다.
    rows = [{"display_order": order, "source_id": doc.metadata["source_id"], "title": doc.metadata["title"],
             **{key: doc.metadata[key] for key in sorted(doc.metadata) if key not in {"source_id", "title", "url"}},
             "text": doc.page_content} for order, doc in enumerate(documents, start=1)]
    with pd.option_context("display.max_columns", None, "display.max_colwidth", None):
        display(pd.DataFrame(rows).head())


아래 준비 셀을 순서대로 실행하세요. 교안 01과 같은 토큰화 함수 `kiwi_tokenize`, 벡터 저장소 `vector_store`, Recall 계산 함수 `source_recall`을 준비합니다.


In [ ]:
# 분석기는 한 번 준비해 문서와 질문 양쪽에 같은 방식으로 적용합니다.
kiwi = Kiwi()


def kiwi_tokenize(text):
    """명사·외국어·숫자를 소문자 토큰 목록으로 돌려줍니다."""
    # PDF 표기 통일: 재택･원격근무 -> 재택·원격근무
    text = text.replace("･", "·")
    # N 계열은 명사, SL은 외국어, SN은 숫자입니다. 조사는 제외합니다.
    # lower는 영문 대소문자를 통일합니다. 기호가 중요한 제품 코드는 별도로 살펴봅니다.
    return [token.form.lower() for token in kiwi.tokenize(text)
            if token.tag.startswith("N") or token.tag in {"SL", "SN"}]


In [ ]:
# 같은 이름의 수업용 메모리 컬렉션만 비웁니다. 재실행하면 문서를 다시 임베딩합니다.
vector_store = Chroma(collection_name="day47_lv1_wiki", embedding_function=embedding_model)
vector_store.reset_collection()
# make_documents가 넣은 Document.id가 저장소 ID가 되므로 원문 ID가 검색 결과까지 그대로 이어집니다.
added_ids = vector_store.add_documents(documents)
print("day47_lv1_wiki 적재 수:", len(added_ids))


In [ ]:
def source_recall(documents, expected_ids):
    """정답 문서 ID 중 검색된 ID의 비율을 반환합니다."""
    expected = set(expected_ids)
    retrieved_ids = {doc.metadata["source_id"] for doc in documents}
    # 분모는 검색 결과 수가 아니라 필요한 원문 수입니다.
    return len(retrieved_ids & expected) / len(expected)


## 1. 한국어 질문을 형태소 토큰으로 바꿉니다

**배경**: 한국어 질문에서 조사를 분리하면 문서와 질문에 있는 같은 명사를 같은 토큰으로 비교할 수 있습니다.

**요구사항**:

- **`query_tokens`**: `question`을 `kiwi_tokenize`로 처리해 토큰 목록을 만드세요.

**확인 기준**: 문자열 목록이며, 조사를 뺀 `데이터베이스` 토큰을 포함합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 문서 검색에 사용하는 토큰화 함수를 질문에도 적용합니다.

세부구현:
1. 코드에 준비된 question의 내용을 확인합니다.
2. kiwi_tokenize를 호출한 결과를 query_tokens에 담습니다.
```

</details>


In [ ]:
# 같은 질문을 두 방식으로 나누어 비교합니다.
question = "관계형 데이터베이스를 설명해 주세요"

# 공백 분리에서는 단어에 붙은 조사가 그대로 남습니다.
space_tokens = question.split()


# ====================================================================
# 문서와 질문에 같은 전처리를 적용해야 같은 토큰끼리 비교할 수 있습니다.
# [작성] question을 kiwi_tokenize에 전달하는 호출식을 작성하세요. 반환값은 문자열 목록입니다.
query_tokens = ...
# ====================================================================


# 두 목록에서 '데이터베이스를'과 '데이터베이스'를 찾아 비교합니다.
print("공백 분리:", space_tokens)
print("형태소 토큰:", query_tokens)


In [ ]:
# [자가채점]
assert isinstance(query_tokens, list) and "데이터베이스" in query_tokens, "query_tokens에서 조사가 분리된 데이터베이스 토큰을 확인하세요."
print("✅ 1번 핵심 확인 완료!")


## 2. 한국어 전처리를 연결한 BM25 검색기를 만듭니다

**배경**: GeoJSON이라는 용어를 설명하는 문서를 찾기 위해 BM25 검색을 실행합니다.

**요구사항**:

- **`bm25`와 `bm25_results`**: 한국어 토큰화 함수를 연결한 BM25 검색기를 만들고, 준비된 `bm25_question`으로 검색하세요.

**확인 기준**: `bm25`는 `kiwi_tokenize`로 전처리하는 BM25 검색기입니다. `bm25_results`에는 검색 문서가 1~2개 담깁니다.

<details><summary>힌트</summary>

```text
접근방법:
- BM25Retriever.from_documents로 문서와 토큰화 함수를 연결한 뒤 검색합니다.

세부구현:
1. preprocess_func에는 함수를 호출하지 말고 함수 자체를 전달합니다.
2. 설정값은 주석을 참고하고, 만든 검색기의 invoke로 질문을 검색합니다.
```

</details>


In [ ]:
# 1) 검색할 한글 질문을 준비합니다.
bm25_question = "GeoJSON 형식을 설명한 문서를 찾아 주세요."

# ====================================================================
# [작성] 2) BM25 검색기를 만들고 질문을 검색하세요.
# BM25Retriever.from_documents의 첫 인자는 documents입니다.
# preprocess_func에는 kiwi_tokenize 함수 자체를 연결합니다.
# bm25_params는 {"k1": 1.5, "b": 0.75}, k는 2로 설정합니다.
# bm25_results에는 만든 검색기의 invoke로 받은 결과를 담습니다.
bm25 = ...
bm25_results = ...
# ====================================================================

# 3) 원문 ID와 본문을 읽어 검색 결과를 확인합니다.
show_results(bm25_results)


In [ ]:
# [자가채점]
assert isinstance(bm25, BM25Retriever) and bm25.preprocess_func is kiwi_tokenize, "BM25Retriever.from_documents에 preprocess_func=kiwi_tokenize를 연결하세요."
assert isinstance(bm25_results, list) and 0 < len(bm25_results) <= 2, "만든 bm25로 질문을 검색해 Document 목록을 받으세요."
print("✅ 2번 핵심 확인 완료!")


## 3. 각 문서의 BM25 점수를 구합니다

**배경**: 검색 결과에 포함된 문서가 질문의 토큰과 실제로 일치했는지 BM25 점수로 확인합니다.

**요구사항**:

- **`bm25_scores`**: 준비된 `score_tokens`로 `bm25.vectorizer.get_scores`를 호출해 문서별 점수를 받으세요.

**확인 기준**: `documents`와 같은 순서로 점수 5개가 담깁니다.

<details><summary>힌트</summary>

```text
접근방법:
- get_scores에는 질문 문자열이 아니라 토큰 목록을 전달합니다.

세부구현:
1. 이미 토큰화된 score_tokens를 사용합니다.
2. get_scores의 반환값을 bm25_scores에 담습니다.
```

</details>


In [ ]:
# get_scores에 전달할 질문 토큰을 미리 준비합니다.
# get_scores는 문자열을 자동 토큰화하지 않습니다. 토큰 목록을 넘겨야 합니다.
score_tokens = kiwi_tokenize("GeoJSON 형식을 설명한 문서를 찾아 주세요.")

# ====================================================================
# [작성] bm25.vectorizer.get_scores에 score_tokens를 전달하는 호출식을 작성하세요.
bm25_scores = ...
# ====================================================================


# 점수는 documents와 같은 순서이므로 원문과 점수를 같은 위치끼리 묶습니다.
# 이 딕셔너리는 결과를 읽기 위한 코드이며 직접 작성하지 않습니다.
score_by_id = {}
for doc, score in zip(documents, bm25_scores):
    source_id = doc.metadata["source_id"]
    score_by_id[source_id] = float(score)

# 점수가 0인 원문도 검색 결과에 포함되는지 확인합니다.
print("원문 ID별 BM25 점수:", score_by_id)
print("검색 결과 ID:", [doc.metadata["source_id"] for doc in bm25_results])


In [ ]:
# [자가채점]
assert len(bm25_scores) == len(documents), "get_scores로 문서마다 점수 하나씩 받았는지 확인하세요."
print("✅ 3번 핵심 확인 완료!")


## 4. Dense로 의미가 가까운 문서를 검색합니다

**배경**: 정확한 용어 대신 풀어 쓴 질문으로 관련 문서를 찾기 위해 Dense 검색을 실행합니다.

**요구사항**:

- **`dense`와 `dense_results`**: `vector_store`를 검색기로 바꾸고, 준비된 `dense_question`으로 의미 검색을 실행하세요.

**확인 기준**: `dense_results`에는 검색한 `Document`가 1~2개 담깁니다.

<details><summary>힌트</summary>

```text
접근방법:
- 벡터 저장소의 as_retriever로 검색기를 만들고 invoke로 질문을 검색합니다.

세부구현:
1. search_kwargs의 k로 반환 개수를 지정합니다.
2. 만든 dense에 dense_question을 전달합니다.
```

</details>


In [ ]:
# 두 검색기에 같은 질문을 전달합니다. 이 질문은 5번에서도 사용합니다.
dense_question = "표 모양으로 정보를 정리해 두고 SQL로 꺼내 보는 시스템"


# ====================================================================
# [작성] vector_store.as_retriever로 검색기를 만들고 질문을 검색하세요.
# search_kwargs에는 {"k": 2}를 전달해 BM25와 같은 개수로 비교합니다.
# dense_results에는 dense.invoke로 dense_question을 검색한 결과를 담습니다.
dense = ...
dense_results = ...
# ====================================================================

bm25_compare_results = bm25.invoke(dense_question)

# 제목 대신 원문 ID로 공통 문서를 찾습니다.
dense_ids = {doc.metadata["source_id"] for doc in dense_results}
bm25_ids = {doc.metadata["source_id"] for doc in bm25_compare_results}
common_ids = dense_ids & bm25_ids

# 검색된 원문을 읽고 어떤 내용을 근거로 찾았는지 비교합니다.
print("Dense 결과")
show_results(dense_results)
print("BM25 결과")
show_results(bm25_compare_results)
print("두 결과에 모두 있는 ID:", common_ids)


In [ ]:
# [자가채점]
assert isinstance(dense_results, list) and 0 < len(dense_results) <= 2, "dense로 검색한 문서 목록을 dense_results에 담으세요."
print("✅ 4번 핵심 확인 완료!")


## 5. BM25와 Dense를 하이브리드 검색기로 묶습니다

**배경**: 단어 일치와 의미 유사도로 찾은 결과를 함께 사용하기 위해 두 검색기를 연결합니다.

**요구사항**:

- **`hybrid`**: `bm25`와 `dense`를 이 순서로 연결한 `EnsembleRetriever` 객체를 만드세요. 설정값은 작성 구간의 주석에 있습니다.

**확인 기준**: `hybrid`는 `bm25`, `dense` 순서로 연결한 `EnsembleRetriever` 객체입니다.

<details><summary>힌트</summary>

```text
접근방법:
- EnsembleRetriever의 retrievers에는 검색기 객체의 목록을 넣습니다.

세부구현:
1. retrievers에 bm25와 dense를 순서대로 연결합니다.
2. 코드 주석의 weights, c, id_key를 지정해 hybrid를 생성합니다.
```

</details>


In [ ]:
# ====================================================================
# [작성] EnsembleRetriever를 생성하는 호출식 전체를 작성하세요.
# retrievers에는 [bm25, dense], weights에는 [0.3, 0.7]을 지정합니다.
# c는 60, id_key는 "source_id"로 설정해 같은 원문을 합칩니다.
hybrid = ...
# ====================================================================


# 4번과 같은 질문으로 하이브리드 검색을 수행합니다.
# 결과는 두 검색기가 찾은 원문의 합집합이므로 2개보다 많을 수 있습니다.
hybrid_results = hybrid.invoke(dense_question)

# 합친 결과의 앞 2개를 선택합니다.
hybrid_top = hybrid_results[:2]

# 전체 후보 수와 최종 선택된 원문을 확인합니다.
print("합친 결과 수:", len(hybrid_results))
show_results(hybrid_top)


In [ ]:
# [자가채점]
assert isinstance(hybrid, EnsembleRetriever), "hybrid는 EnsembleRetriever로 만드세요."
assert hybrid.retrievers == [bm25, dense], "retrievers에 bm25, dense 순서로 두 검색기를 연결하세요."
print("✅ 5번 핵심 확인 완료!")


## 6. 두 검색기의 비중이 같은 하이브리드 검색기를 만듭니다

**배경**: 같은 검색 후보에 가중치만 다르게 적용해 최종 순위가 달라지는지 비교합니다.

**요구사항**:

- **`hybrid_even`**: 5번과 같은 `bm25`·`dense`를 연결하고, 가중치를 `[0.5, 0.5]`로 지정한 `EnsembleRetriever` 객체를 만드세요.

**확인 기준**: `hybrid_even.weights`가 `[0.5, 0.5]`입니다. 가중치를 바꿔도 출력 순위와 Recall@2가 같을 수 있습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 5번의 검색기 구성에서 가중치만 바꿉니다.

세부구현:
1. 두 검색기를 같은 순서로 연결하는 EnsembleRetriever를 새로 만듭니다.
2. weights를 동일한 비중으로 지정하고, 나머지 설정은 유지합니다.
```

</details>


In [ ]:
# 비교할 질문과 정답 원문 ID입니다. 두 가중치에서 같은 값을 사용합니다.
compare_question = "코미디 프로그램에서 이름을 딴 언어와 행과 열로 자료를 저장하는 시스템을 알려 주세요"
expected_ids = {"wiki0", "wiki1"}

# 두 검색기에 같은 compare_question을 사용합니다.
# candidates[0]은 BM25 결과, candidates[1]은 Dense 결과인 중첩 목록입니다.
# 두 검색은 여기서 한 번씩만 하고, 아래에서는 이 후보를 그대로 사용합니다.
candidates = [
    bm25.invoke(compare_question),
    dense.invoke(compare_question),
]


# ====================================================================
# [작성] 두 검색기의 비중이 같은 EnsembleRetriever를 생성하세요.
# retrievers에는 [bm25, dense], weights에는 [0.5, 0.5]를 지정합니다.
# 5번과 같은 c=60, id_key="source_id" 설정을 유지합니다.
hybrid_even = ...
# ====================================================================


# 비교 결과를 모으는 딕셔너리입니다. 이 블록은 그대로 사용하세요.
# BM25·Dense는 기존 후보를, 두 Hybrid는 같은 후보를 각 가중치로 합친 결과를 담습니다.
# weighted_reciprocal_rank는 새로 검색하지 않고, 주어진 목록의 순위를 합칩니다.
method_results = {
    "BM25": candidates[0][:2],
    "Dense": candidates[1][:2],
    "Hybrid 0.3/0.7": hybrid.weighted_reciprocal_rank(candidates)[:2],
    "Hybrid 0.5/0.5": hybrid_even.weighted_reciprocal_rank(candidates)[:2],
}

# 필요한 원문 2개 중 몇 개를 찾았는지 계산합니다.
# name은 방식 이름, results는 해당 방식의 최종 원문 목록입니다.
recall_by_method = {}
for name, results in method_results.items():
    recall_by_method[name] = source_recall(results, expected_ids)

# 방식별 원문 ID와 Recall@2를 한 줄씩 비교합니다.
for name, results in method_results.items():
    source_ids = [doc.metadata["source_id"] for doc in results]
    print(name, "/ 원문 ID:", source_ids, "/ Recall@2:", recall_by_method[name])


In [ ]:
# [자가채점]
assert hybrid_even.weights == [0.5, 0.5], "hybrid_even의 weights를 [0.5, 0.5]로 지정하세요."
print("✅ 6번 핵심 확인 완료!")


## 7. RRF가 원래 점수 대신 순위를 쓰는 이유를 설명합니다

**배경**: BM25와 Dense의 결과를 하나로 합칠 때 두 검색기의 점수를 그대로 더해도 되는지 판단해야 합니다.

**요구사항**:

- **설명**: RRF가 두 검색기의 원래 점수를 바로 더하지 않고 순위를 사용하는 이유를 1~2문장으로 쓰세요.

**확인 기준**: 두 검색기의 점수를 바로 더하기 어려운 이유와 순위를 쓰는 장점을 연결하면 됩니다. 공식이나 계산은 필요하지 않습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 같은 숫자가 두 검색기에서 같은 의미인지 생각합니다.

세부구현:
1. BM25 점수와 Dense의 유사도·거리가 어떻게 다른지 떠올립니다.
2. 순위를 공통 기준으로 쓰는 이유를 설명합니다.
```

</details>


*(여기에 판단과 근거를 서술하세요)*
